# Kenya Climate Data Analysis

## Purpose
This notebook focuses specifically on Kenya's climate patterns as part of the regional analysis strategy.
**Why we're analyzing Kenya separately:**
- Kenya has diverse climate zones (coastal, highlands, arid north)
- Complex topography creates microclimate variations
- Prevents skewed averages in multi-country datasets
- Enables targeted climate policy recommendations for Kenya's agriculture and tourism

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Set style for professional visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

## Data Loading - Kenya Specific

**Intent:** Load only Kenya data to ensure focused analysis without cross-country contamination.

In [ ]:
# Load Kenya-specific climate data
df_kenya = pd.read_csv('../data/kenya.csv')

# Filter to ensure we only have Kenya data (quality check)
print(f"Dataset shape: {df_kenya.shape}")
print(f"Date range: {df_kenya['YEAR'].min()} to {df_kenya['YEAR'].max()}")
print(f"Total records: {len(df_kenya)}")

In [ ]:
# Data profiling for Kenya
print("=== KENYA CLIMATE DATA PROFILING ===")
print("\nData Info:")
df_kenya.info()

print("\nDescriptive Statistics:")
print(df_kenya[['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'RH2M', 'WS2M']].describe())

## Data Cleaning - Kenya Specific

**Why we're cleaning Kenya data separately:**
- Kenya's diverse climate zones may have different outlier patterns
- Highland and coastal variations require country-specific thresholds
- Ensures Kenya's climate patterns aren't masked by other countries

In [ ]:
# Check for missing values in Kenya data
print("Missing values in Kenya dataset:")
missing_values = df_kenya.isnull().sum()
print(missing_values[missing_values > 0])

# Check for invalid values (-999) common in climate data
invalid_counts = {}
for col in ['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'RH2M', 'WS2M']:
    invalid_count = (df_kenya[col] == -999).sum()
    if invalid_count > 0:
        invalid_counts[col] = invalid_count

print(f"\nInvalid values (-999) in Kenya data: {invalid_counts}")

In [ ]:
# Clean Kenya data
df_kenya_clean = df_kenya.copy()

# Replace invalid values with NaN
climate_columns = ['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'RH2M', 'WS2M']
for col in climate_columns:
    df_kenya_clean[col] = df_kenya_clean[col].replace(-999, np.nan)

# Remove rows with critical missing data
initial_count = len(df_kenya_clean)
df_kenya_clean = df_kenya_clean.dropna(subset=['T2M', 'PRECTOTCORR'])
final_count = len(df_kenya_clean)

print(f"Kenya data cleaning results:")
print(f"Initial records: {initial_count}")
print(f"After removing invalid/critical missing data: {final_count}")
print(f"Records removed: {initial_count - final_count} ({((initial_count - final_count)/initial_count)*100:.2f}%)")

## Kenya Temperature Analysis

**Focus:** Understanding Kenya's diverse temperature patterns across different climate zones.

In [ ]:
# Kenya temperature trends
plt.figure(figsize=(15, 10))

# Create date column for time series
df_kenya_clean['Date'] = pd.to_datetime(df_kenya_clean['YEAR'].astype(str) + '-' + 
                                      df_kenya_clean['DOY'].astype(str), format='%Y-%j')

# Plot 1: Daily Temperature Trends
plt.subplot(2, 2, 1)
plt.plot(df_kenya_clean['Date'], df_kenya_clean['T2M'], alpha=0.6, color='green')
plt.title('Kenya Daily Temperature Trends (2015-2026)')
plt.xlabel('Date')
plt.ylabel('Temperature (°C)')
plt.xticks(rotation=45)

# Plot 2: Monthly Temperature Distribution
plt.subplot(2, 2, 2)
monthly_temp = df_kenya_clean.groupby(df_kenya_clean['Date'].dt.month)['T2M'].mean()
monthly_temp.plot(kind='bar', color='orange')
plt.title('Kenya Average Monthly Temperature')
plt.xlabel('Month')
plt.ylabel('Temperature (°C)')
plt.xticks(range(12), ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
                    'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'], rotation=45)

# Plot 3: Temperature Range Analysis
plt.subplot(2, 2, 3)
temp_range = df_kenya_clean['T2M_MAX'] - df_kenya_clean['T2M_MIN']
plt.hist(temp_range.dropna(), bins=30, color='green', alpha=0.7)
plt.title('Kenya Daily Temperature Range Distribution')
plt.xlabel('Temperature Range (°C)')
plt.ylabel('Frequency')

# Plot 4: Yearly Temperature Trends
plt.subplot(2, 2, 4)
yearly_temp = df_kenya_clean.groupby('YEAR')['T2M'].mean()
plt.plot(yearly_temp.index, yearly_temp.values, marker='o', color='darkgreen')
plt.title('Kenya Yearly Average Temperature')
plt.xlabel('Year')
plt.ylabel('Temperature (°C)')

plt.tight_layout()
plt.show()

# Print key statistics
print("=== KENYA TEMPERATURE ANALYSIS ===")
print(f"Average Temperature: {df_kenya_clean['T2M'].mean():.2f}°C")
print(f"Max Temperature: {df_kenya_clean['T2M_MAX'].max():.2f}°C")
print(f"Min Temperature: {df_kenya_clean['T2M_MIN'].min():.2f}°C")
print(f"Average Daily Range: {temp_range.mean():.2f}°C")
print(f"Temperature Stability: {df_kenya_clean['T2M'].std():.2f}°C (lower = more stable)")

## Kenya Rainfall Analysis

**Purpose:** Analyze Kenya's bimodal rainfall patterns crucial for agricultural planning and water resource management.

In [ ]:
# Kenya rainfall analysis
plt.figure(figsize=(15, 10))

# Plot 1: Daily Rainfall
plt.subplot(2, 2, 1)
plt.plot(df_kenya_clean['Date'], df_kenya_clean['PRECTOTCORR'], alpha=0.6, color='blue')
plt.title('Kenya Daily Rainfall (2015-2026)')
plt.xlabel('Date')
plt.ylabel('Rainfall (mm/day)')
plt.xticks(rotation=45)

# Plot 2: Monthly Rainfall Patterns (Bimodal pattern)
plt.subplot(2, 2, 2)
monthly_rain = df_kenya_clean.groupby(df_kenya_clean['Date'].dt.month)['PRECTOTCORR'].mean()
monthly_rain.plot(kind='bar', color='cyan')
plt.title('Kenya Average Monthly Rainfall (Bimodal Pattern)')
plt.xlabel('Month')
plt.ylabel('Rainfall (mm/day)')
plt.xticks(range(12), ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
                    'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'], rotation=45)

# Plot 3: Rainfall Distribution
plt.subplot(2, 2, 3)
plt.hist(df_kenya_clean['PRECTOTCORR'].dropna(), bins=50, color='purple', alpha=0.7)
plt.title('Kenya Daily Rainfall Distribution')
plt.xlabel('Rainfall (mm/day)')
plt.ylabel('Frequency')
plt.yscale('log')  # Log scale due to many zero rainfall days

# Plot 4: Yearly Rainfall Totals
plt.subplot(2, 2, 4)
yearly_rain = df_kenya_clean.groupby('YEAR')['PRECTOTCORR'].sum()
plt.plot(yearly_rain.index, yearly_rain.values, marker='s', color='darkblue')
plt.title('Kenya Yearly Total Rainfall')
plt.xlabel('Year')
plt.ylabel('Total Rainfall (mm/year)')

plt.tight_layout()
plt.show()

# Print key statistics
print("=== KENYA RAINFALL ANALYSIS ===")
print(f"Average Daily Rainfall: {df_kenya_clean['PRECTOTCORR'].mean():.2f} mm/day")
print(f"Max Daily Rainfall: {df_kenya_clean['PRECTOTCORR'].max():.2f} mm/day")
print(f"Rainy Days (>0.1mm): {(df_kenya_clean['PRECTOTCORR'] > 0.1).sum()}")
print(f"Dry Days: {(df_kenya_clean['PRECTOTCORR'] <= 0.1).sum()}")
print(f"Average Annual Rainfall: {yearly_rain.mean():.2f} mm/year")

# Identify bimodal rainfall pattern
peak_months = monthly_rain.nlargest(2)
print(f"\nBimodal Rainfall Pattern - Peak Months:")
for month, rainfall in peak_months.items():
    month_names = ['', 'Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    print(f"  {month_names[month]}: {rainfall:.2f} mm/day")

## Kenya Climate Anomalies Detection

**Why this matters for Kenya:**
- Kenya's diverse climate zones show different anomaly patterns
- Early detection supports climate adaptation for agriculture and tourism
- Regional specificity improves policy relevance

In [ ]:
# Detect temperature anomalies in Kenya
temp_mean = df_kenya_clean['T2M'].mean()
temp_std = df_kenya_clean['T2M'].std()

# Define anomalies (±2 standard deviations)
temp_upper_threshold = temp_mean + 2 * temp_std
temp_lower_threshold = temp_mean - 2 * temp_std

hot_days = df_kenya_clean[df_kenya_clean['T2M'] > temp_upper_threshold]
cold_days = df_kenya_clean[df_kenya_clean['T2M'] < temp_lower_threshold]

print("=== KENYA TEMPERATURE ANOMALIES ===")
print(f"Temperature Mean: {temp_mean:.2f}°C")
print(f"Temperature Std Dev: {temp_std:.2f}°C")
print(f"Upper Threshold: {temp_upper_threshold:.2f}°C")
print(f"Lower Threshold: {temp_lower_threshold:.2f}°C")
print(f"\nHot Days (>2σ): {len(hot_days)} ({len(hot_days)/len(df_kenya_clean)*100:.2f}%)")
print(f"Cold Days (<-2σ): {len(cold_days)} ({len(cold_days)/len(df_kenya_clean)*100:.2f}%)")

# Detect rainfall anomalies (important for Kenya's bimodal pattern)
rain_mean = df_kenya_clean['PRECTOTCORR'].mean()
rain_std = df_kenya_clean['PRECTOTCORR'].std()

heavy_rain_threshold = rain_mean + 2 * rain_std
heavy_rain_days = df_kenya_clean[df_kenya_clean['PRECTOTCORR'] > heavy_rain_threshold]

print(f"\n=== KENYA RAINFALL ANOMALIES ===")
print(f"Rainfall Mean: {rain_mean:.2f} mm/day")
print(f"Heavy Rain Threshold (>2σ): {heavy_rain_threshold:.2f} mm/day")
print(f"Heavy Rain Days: {len(heavy_rain_days)} ({len(heavy_rain_days)/len(df_kenya_clean)*100:.2f}%)")

# Seasonal anomaly analysis
print(f"\n=== KENYA SEASONAL ANALYSIS ===")
long_rains = monthly_rain.loc[[3, 4, 5]].sum()  # March-May
short_rains = monthly_rain.loc[[10, 11, 12]].sum()  # Oct-Dec
dry_season = monthly_rain.loc[[6, 7, 8, 9]].sum()  # Jun-Sep

print(f"Long Rains (Mar-May): {long_rains:.2f} mm/day")
print(f"Short Rains (Oct-Dec): {short_rains:.2f} mm/day")
print(f"Dry Season (Jun-Sep): {dry_season:.2f} mm/day")
print(f"Rainfall Seasonality Ratio: {long_rains/short_rains:.2f} (Long/Short)")

## Kenya Climate Summary & Policy Insights

**Key findings for Kenya's climate policy planning:**

In [ ]:
# Generate Kenya-specific climate summary
kenya_summary = {
    'temperature_profile': {
        'mean_temp': df_kenya_clean['T2M'].mean(),
        'temp_variability': df_kenya_clean['T2M'].std(),
        'extreme_hot_days': len(hot_days),
        'extreme_cold_days': len(cold_days),
        'temp_stability': df_kenya_clean['T2M'].std()
    },
    'rainfall_profile': {
        'mean_daily_rainfall': df_kenya_clean['PRECTOTCORR'].mean(),
        'annual_total_rainfall': yearly_rain.mean(),
        'bimodal_pattern': True,
        'long_rains_total': long_rains,
        'short_rains_total': short_rains,
        'heavy_rain_events': len(heavy_rain_days)
    },
    'seasonal_patterns': {
        'peak_rain_months': peak_months.index.tolist(),
        'dry_season_length': 4,  # June-September
        'rainfall_consistency': monthly_rain.std()
    },
    'data_quality': {
        'total_records': len(df_kenya_clean),
        'data_completeness': (1 - df_kenya_clean.isnull().sum().sum() / (len(df_kenya_clean) * len(climate_columns))) * 100,
        'years_covered': df_kenya_clean['YEAR'].nunique()
    }
}

print("=== KENYA CLIMATE SUMMARY FOR POLICY MAKERS ===")
for category, metrics in kenya_summary.items():
    print(f"\n{category.upper().replace('_', ' ')}:")
    for metric, value in metrics.items():
        if isinstance(value, float):
            print(f"  {metric.replace('_', ' ').title()}: {value:.2f}")
        else:
            print(f"  {metric.replace('_', ' ').title()}: {value}")

# Policy recommendations for Kenya
print("\n=== KENYA-SPECIFIC POLICY RECOMMENDATIONS ===")
if kenya_summary['temperature_profile']['temp_stability'] < 3:
    print("✅ STABLE TEMPERATURES: Favorable for consistent agricultural planning")
if kenya_summary['rainfall_profile']['bimodal_pattern']:
    print("🌧️ BIMODAL RAINFALL: Plan agriculture around long and short rain seasons")
if kenya_summary['rainfall_profile']['annual_total_rainfall'] > 800:
    print("💧 ADEQUATE RAINFALL: Support rain-fed agriculture development")
if kenya_summary['rainfall_profile']['heavy_rain_events'] > len(df_kenya_clean) * 0.05:
    print("⚠️ FLOOD RISK: Develop flood management for highland regions")

print("\n✅ Kenya climate analysis completed successfully!")
print("Key insight: Kenya's stable temperatures and bimodal rainfall support diverse agriculture.")